# Verificación Física de `Dirac-Solver`

> **Idioma / Language:** **Español** | [English](../en/06_physics_verification.ipynb)

Este tutorial valida la fidelidad física y numérica del integrador C++ FDTD Leapfrog mediante tres pruebas canónicas:
1. **Evolución de fase de una onda plana libre:** $\psi(x, t) = \psi(x, 0) e^{-i E t}$ con $E = \sqrt{p_x^2 + m^2}$.
2. **Conservación unitaria de la probabilidad total:** $P(t) = \int \psi^\dagger \psi \, dV = \text{const.}$
3. **Conservación del valor esperado de la energía:** $\langle H \rangle(t) = \int \psi^\dagger \hat{H} \psi \, dV = \text{const.}$

In [ ]:
import sys
sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from dirac_solver import (
    Grid, ConstantSpinor, PlaneWave, GaussianPacket,
    DiracProblemBuilder, DiracSolver, FreeParticle, Observables, _core
)

## 1. Evolución de Fase de una Onda Plana Libre

Para una partícula libre en 1D con momento $p_x = 1.0$ y masa $m = 1.0$, la solución analítica exacta satisface $\psi(x, t) = \psi(x, 0) e^{-i\sqrt{p_x^2 + m^2}\, t}$.

In [ ]:
L = 10.0
N = 200
dx = L / N
grid = Grid(shape=(N,), spacing=(dx,))
print(f'Límite de Courant dt_max = {grid.courant_limit():.5f}')

px, mass = 1.0, 1.0
spinor = ConstantSpinor(momentum=[px, 0.0, 0.0], mass=mass)
initial_state = PlaneWave(spinor)

dt = 0.0005
total_time = 0.2

problem = (
    DiracProblemBuilder()
    .set_grid(grid)
    .set_initial_state(initial_state)
    .set_potential(FreeParticle())
    .set_boundary_condition(_core.PeriodicBoundary())
    .set_time_parameters(time_step=dt, total_time=total_time)
    .build()
)

solver = DiracSolver(problem)
psi_0 = solver.get_psi().copy()
solver.run_simulation()
psi_t = solver.get_psi()

E = np.sqrt(px**2 + mass**2)
psi_analytical = psi_0 * np.exp(-1j * E * total_time)
rel_error = np.linalg.norm(psi_t - psi_analytical) / np.linalg.norm(psi_analytical)
print(f'Error relativo respecto a la solución analítica: {rel_error:.6e}')

## 2. Conservación de la Probabilidad Total y de la Energía Esperada

Evolucionamos un paquete gaussiano relativista y registramos $P(t)$ y $\langle H \rangle(t)$ en cada paso.

In [ ]:
grid_gp = Grid(shape=(200,), spacing=(0.1,))
spinor_gp = ConstantSpinor(momentum=[1.5, 0.0, 0.0], mass=1.0)
packet = GaussianPacket(spinor_gp, center=[0.0, 0.0, 0.0], spatial_width=1.5)

problem_gp = (
    DiracProblemBuilder()
    .set_grid(grid_gp)
    .set_initial_state(packet)
    .set_potential(FreeParticle())
    .set_boundary_condition(_core.PeriodicBoundary())
    .set_time_parameters(time_step=0.001, total_time=1.0)
    .build()
)

solver_gp = DiracSolver(problem_gp)
obs = Observables(solver_gp)
solver_gp.run_simulation(observables_handler=obs)

times = obs.get_history('time')
rho_hist = obs.get_history('probability_density')
prob_total = np.sum(rho_hist, axis=1) * grid_gp.get_cell_volume()
energy_hist = obs.get_history('expected_energy')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(times, prob_total / prob_total[0])
axes[0].set_title('Probabilidad Total Normalizada $P(t)/P(0)$')
axes[0].set_xlabel('Tiempo $t$')
axes[0].grid(True)

axes[1].plot(times, energy_hist)
axes[1].set_title('Valor Esperado de la Energía $\\langle H \\rangle(t)$')
axes[1].set_xlabel('Tiempo $t$')
axes[1].grid(True)
plt.tight_layout()
plt.show()

print(f'Deriva máxima relativa de probabilidad: {np.max(np.abs(prob_total - prob_total[0])) / prob_total[0]:.4e}')
print(f'Deriva máxima relativa de energía:      {np.max(np.abs(energy_hist - energy_hist[0])) / abs(energy_hist[0]):.4e}')